# 10. Мини-проект

## Постановка задачи

К вам попали результаты A/A/B-тестирования от одного известного маркетплейса. 

> *sample_a, sample_c — АА-группы*
> 
> *sample_b — отдельная группа.*

В каждом датасете есть три типа действий пользователей: 0 — клик, 1 — просмотр и 2 — покупка (пользователь просматривает выдачу товаров, кликает на понравившийся товар и совершает покупку).

Маркетплейс ориентируется на следующие метрики:

- ctr (отношение кликов к просмотрам товаров);
- purchase rate (отношение покупок к просмотрам товаров);
- gmv (оборот, сумма произведений количества покупок на стоимость покупки), где считаем 1 сессию за 1 точку (1 сессия на 1 пользователя).

Данные уже почищены по сессиям, вы можете использовать их в агрегированном виде. Ваша задача — понять, нет ли проблемы с разъезжанием сплитов и улучшает ли алгоритм B работу маркетплейса.

Тест Шапиро-Уилка проведите на **alpha = 0.01**

**Критерии оценивания:**

- 2 балла	Проведена проверка, что нет ситуаций, когда происходит покупка/клик без действия просмотра. Удалены дубли.
- 2 балла	Рассчитаны метрики по датасетам, проведено общее сравнение метрик.
- 3 балла	Проведён тест равенства долей для A и C групп по всем метрикам. Сделан вывод.
- 3 балла	Проведён тест равенства долей для A и B групп по всем метрикам. Сделан вывод

**Подсказки**
1. Что важно как требование к чистоте данных? 
2. Посмотрите внимательно на значения метрик!
3. «Разъезжаются» ли сплиты? Посмотрите на результаты A/A-теста.
4. Каков результат А/B-теста? можем ли мы на него положиться?

In [34]:
import pandas as pd

## Загрузим данные

In [35]:
df_sample_a = pd.read_csv('data_mvp3/sample_a.csv')

print(df_sample_a.head(5))
print()
print(df_sample_a.info())

   user_id  item_id  action_id
0    84636      360          1
1    21217     9635          1
2    13445     8590          1
3    38450     5585          1
4    14160     2383          0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1188912 entries, 0 to 1188911
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1188912 non-null  int64
 1   item_id    1188912 non-null  int64
 2   action_id  1188912 non-null  int64
dtypes: int64(3)
memory usage: 27.2 MB
None


In [36]:
df_sample_b = pd.read_csv('data_mvp3/sample_b.csv')

print(df_sample_b.head(5))
print()
print(df_sample_b.info())

   user_id  item_id  action_id
0   118375     4105          1
1   107569     8204          1
2   175990      880          1
3   160582     9568          0
4   123400     4000          1

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1198438 entries, 0 to 1198437
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1198438 non-null  int64
 1   item_id    1198438 non-null  int64
 2   action_id  1198438 non-null  int64
dtypes: int64(3)
memory usage: 27.4 MB
None


In [37]:
df_sample_c = pd.read_csv('data_mvp3/sample_c.csv')

print(df_sample_c.head(5))
print()
print(df_sample_c.info())

   user_id  item_id  action_id
0   274623     2863          1
1   265472      343          1
2   242779     6009          0
3   275009     2184          1
4   268104     3134          2

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1205510 entries, 0 to 1205509
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1205510 non-null  int64
 1   item_id    1205510 non-null  int64
 2   action_id  1205510 non-null  int64
dtypes: int64(3)
memory usage: 27.6 MB
None


In [38]:
df_item_prices = pd.read_csv('data_mvp3/item_prices.csv')
print(df_item_prices.head(5))
print()
print(df_item_prices.info())

   item_id  item_price
0      338        1501
1       74         647
2     7696         825
3      866         875
4     5876         804

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   item_id     1000 non-null   int64
 1   item_price  1000 non-null   int64
dtypes: int64(2)
memory usage: 15.8 KB
None


## Проверим на дубликаты

In [39]:
datasets = {
    'df_sample_a': df_sample_a,
    'df_sample_b': df_sample_b,
    'df_sample_c': df_sample_c,
    'df_item_prices': df_item_prices
}

for name, df in datasets.items():
    dup_count = df.duplicated().sum()
    print(f"{name}: {dup_count} дубликатов")
    
    if dup_count > 0:
        print("Первые дубли:")
        print(df[df.duplicated(keep=False)].head())

df_sample_a: 0 дубликатов
df_sample_b: 0 дубликатов
df_sample_c: 0 дубликатов
df_item_prices: 0 дубликатов
